# Modèles décisionnels

Ce notebook présente l'influence des modèles de décision sur le comportement des simulations et donc sur l'estimation des émissions via Copert.

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade sys
%pip install --upgrade os

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade numpy
%pip install --upgrade pandas

%pip install --upgrade shapely

%pip install --upgrade ast

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import sys
import os

from pathlib import Path

import importlib

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import wkt
from shapely.geometry import Point
from shapely.geometry import Polygon

import ast

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes et fonctions

In [ ]:
# import des classes

import modules.CopertEstimator
import modules.DemandVariationCarSimulation1Res
import modules.Sensitivity

importlib.reload(sys.modules['modules.CopertEstimator'])
importlib.reload(sys.modules['modules.DemandVariationCarSimulation1Res'])
importlib.reload(sys.modules['modules.Sensitivity'])

CopertEstimator = modules.CopertEstimator.CopertEstimator
DemandVariationCarSimulation1Res = modules.DemandVariationCarSimulation1Res.DemandVariationCarSimulation1Res
Sensitivity = modules.Sensitivity.Sensitivity

## 2 - Contexte

MnMS propose plusieurs modèle de décision pour le choix de trajets : 
- *DummyDecisionModel*, modèle déterministe ou le trajet au cout le plus faible est toujours choisi;
- *LogitDecisionModel*, modèle probabiliste dans lequel les chances de choisir un trajet dépend du cout de tous les choix possibles;
- *ModeCentricLogitDecisionModel*, modèle déterministe pour chaque mode d'abords avec choix du trajet au cout le plus faible puis modèle probabiliste pour chaque trajet obtenu à l'étape précédente.

Il paraît essentiel de s'intéresser à l'influence de ces modèles sur les simulations et donc sur l'estimation des émissions. Dans les parties suivantes sont effectuées 20 simulations pour chaque combinaison type de demande / modèle de décision. Les émissions de chacun des trafic simulés seront ensuite calculées avec Copert pour les paramètres par défaut (6 minutes pour l'aggrégation temporelle et calcul de la vitesse moyenne via MFD). Pour chaque combinaison sera finalement visualisé et mesuré l'importance des variations des quantités d'émissions calculées.

Les fichiers réseaux seront choisis arbitrairement sauf en ce qui concerne le fichier de description du graphe (fichier de configuration d'une simulation MATSim sur Amsterdam) : leur définition a un impact sur la simulation et ses résultats mais cette influence est absolue, une analyse de sensibilité portant sur le relatif, le choix d'un certain ensemble de fichiers réseaux spécifique n'a donc pas de justification, il est en revanche essentiel une fois l'ensemble déterminer de conserver le même.

Seules les modèles de décision *DummyDecisionModel* et *ModeCentricLogitDecisionModel* seront testés : le modèle ** est plus complexe et il ne paraît pas pertinent de s'attarder sur celle-ci dans le cadre de l'analyse de sensibilité. 

## 3 - Simulations

In [ ]:
# fichiers reseaux

graph_path = "../../network/graph/Amsterdam__Matsim__graph.json"
od_layer_path = "../../network/test_files/Amsterdam__Matsim__graph_od_layer_14_16_1000.json"
transit_link_path = "../../network/test_files/Amsterdam__Matsim__graph_transit_link_14_16_1000_grid.json"
a = 2.8981940480441857
b = -0.00010553060526140915
MFD_variables = [a, b]

### 3.1 - Originaux

In [ ]:
# simulations pour la demande MATSim calibrée

chemin_demande_matsim_originale = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original"
output_directory = "../../outputs/decision_models/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_matsim_originale, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

In [ ]:
# simulations pour la demande Amsterdam originale

chemin_demande_amsterdam_originale = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__original"
output_directory = "../../outputs/decision_models/simulations/AmsterdamCity_7-9_Original__Copies__Simulation_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_amsterdam_originale, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

### 3.1 - DummyDecisionModel

In [ ]:
# simulations pour la demande MATSim calibrée

chemin_demande_matsim_copies = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Copies"
output_directory = "../../outputs/decision_models/simulations/Matsim_7-9_Calibration__Copies__Simulation_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_matsim_copies, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

In [ ]:
# simulations pour la demande Amsterdam originales

chemin_demande_amsterdam_copies = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Copies"
output_directory = "../../outputs/decision_models/simulations/AmsterdamCity_7-9_Original__Copies__Simulation_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_amsterdam_copies, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

### 3.2 - LogitDecisionModel

In [ ]:
# simulations pour la demande MATSim calibrée

chemin_demande_matsim_copies = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Copies"
output_directory = "../../outputs/decision_models/simulations/Matsim_7-9_Calibration__Copies__Simulation_Logit__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_amsterdam_copies, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Logit", output_directory)

In [ ]:
# simulations pour la demande Amsterdam originale

chemin_demande_amsterdam_copies = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Copies"
output_directory = "../../outputs/decision_models/simulations/AmsterdamCity_7-9_Original__Copies__Simulation_Logit__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_matsim_copies, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Logit", output_directory)

## 4 - Copert

In [ ]:
# fonctions affichage 

def plot_emissions(sensitivity_df):
    """
    Plot Nox et Co2 emission estimations on a scatterplot.

    Parameters
    ----------
    sensitivity_df : pandas.DataFrame
        Dataframe to display the Nox and Co2 estimations from.
    """

    buffer = sensitivity_df["ID"] = [i+1 for i in range(len(sensitivity_df))]

    fig, ax = plt.subplots(1,2, figsize=(10,6))

    ax[0].scatter(sensitivity_df["ID"], sensitivity_df["NOX_RATIO"], color="blue")
    ax[0].set_xlabel("ID")
    ax[0].set_ylabel("Nox ratio")
    ax[0].grid(True)

    ax[1].scatter(sensitivity_df["ID"], sensitivity_df["CO2_RATIO"], color="red")
    ax[1].set_xlabel("ID")
    ax[1].set_ylabel("Nox ratio")
    ax[1].grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
# tableaux 


S = Sensitivity()

# Matsim_7-9

original_outputs_directory = "../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00"

S.create_sensitivity_df_from_simulation_outputs_directory(original_outputs_directory=original_outputs_directory,
                                                                         simulation_outputs_directory="../../outputs/decision_models/simulations/Matsim_7-9_Calibration__Copies__Simulation_Dummy__14_16_1000", 
                                                                         period=None, 
                                                                         area=None, 
                                                                         mobility_services=None, 
                                                                         crs="epsg:32631", 
                                                                         copert_data_path="../../data/copert_data/test/FE_COPERT2025.csv", 
                                                                         time_period="6min", 
                                                                         method="SPEED_MEAN")
Matsim_Dummy = S.get_sensitivity_df()

S.create_sensitivity_df_from_simulation_outputs_directory(original_outputs_directory=original_outputs_directory,
                                                                         simulation_outputs_directory="../../outputs/decision_models/simulations/Matsim_7-9_Calibration__Copies__Simulation_Logit__14_16_1000", 
                                                                         period=None, 
                                                                         area=None, 
                                                                         mobility_services=None, 
                                                                         crs="epsg:32631", 
                                                                         copert_data_path="../../data/copert_data/test/FE_COPERT2025.csv", 
                                                                         time_period="6min", 
                                                                         method="SPEED_MEAN")
Matsim_Logit = S.get_sensitivity_df()

# AmsterdamCity_7-9

original_outputs_directory = "../../outputs/AmsterdamCity_7-9/simulations/AmsterdamCity_7-9_Original__Original__Simulation_Dummy__14_16_1000/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__0.00__0_0.00"

S.create_sensitivity_df_from_simulation_outputs_directory(original_outputs_directory=original_outputs_directory,
                                                                         simulation_outputs_directory="../../outputs/decision_models/simulations/AmsterdamCity_7-9_Original__Copies__Simulation_Dummy__14_16_1000", 
                                                                         period=None, 
                                                                         area=None, 
                                                                         mobility_services=None, 
                                                                         crs="epsg:32631", 
                                                                         copert_data_path="../../data/copert_data/test/FE_COPERT2025.csv", 
                                                                         time_period="6min", 
                                                                         method="SPEED_MEAN")


Amsterdam_Dummy = S.get_sensitivity_df()

S.create_sensitivity_df_from_simulation_outputs_directory(original_outputs_directory=original_outputs_directory,
                                                                         simulation_outputs_directory="../../outputs/decision_models/simulations/AmsterdamCity_7-9_Original__Copies__Simulation_Logit__14_16_1000", 
                                                                         period=None, 
                                                                         area=None, 
                                                                         mobility_services=None, 
                                                                         crs="epsg:32631", 
                                                                         copert_data_path="../../data/copert_data/test/FE_COPERT2025.csv", 
                                                                         time_period="6min", 
                                                                         method="SPEED_MEAN")
Amsterdam_Logit = S.get_sensitivity_df()

In [ ]:
Matsim_Dummy.to_csv("../../outputs/decision_models/analysis/Matsim_7-9_Calibration__Original__Dummy_sensitivity.csv", sep=';')
Matsim_Logit.to_csv("../../outputs/decision_models/analysis/Matsim_7-9_Calibration__Original__Logit_sensitivity.csv", sep=';')
Amsterdam_Dummy.to_csv("../../outputs/decision_models/analysis/Amsterdam_7-9_Original__Original__Dummy_sensitivity_df.csv", sep=';')
Amsterdam_Logit.to_csv("../../outputs/decision_models/analysis/Amsterdam_7-9_Original__Original__Logit_sensitivity_df.csv", sep=';')

Matsim_Dummy = pd.read_csv("../../outputs/decision_models/analysis/Matsim_7-9_Calibration__Original__Dummy_sensitivity.csv", sep=';')
Matsim_Logit = pd.read_csv("../../outputs/decision_models/analysis/Matsim_7-9_Calibration__Original__Logit_sensitivity.csv", sep=';')
Amsterdam_Dummy = pd.read_csv("../../outputs/decision_models/analysis/Amsterdam_7-9_Original__Original__Dummy_sensitivity_df.csv", sep=';')
Amsterdam_Logit = pd.read_csv("../../outputs/decision_models/analysis/Amsterdam_7-9_Original__Original__Logit_sensitivity_df.csv", sep=';')

In [ ]:
# affichage Matsim_Dummy

plot_emissions(Matsim_Dummy)

In [ ]:
# affichage Matsim_Logit

plot_emissions(Matsim_Logit)

In [ ]:
# affichage Amsterdam_Dummy

plot_emissions(Amsterdam_Dummy)

In [ ]:
# affichage Amsterdam_Logit

plot_emissions(Amsterdam_Logit)

# 5 - Bilan

Assez logiquement, seul le modèle de decision *DummyDecisionModel* conduit à l'obtention d'estimation des émissions avec variance nulle. Cela peut être simplement expliqué par les mécanismes de fonctionnement de chaque modèle : contrairement aux modèles logit, dont le choix d'un trajet est influencé par les probabilités, le modèle *DummyDecisionModel* choisit le trajet dont le cout est minimal ce qui permet l'obtention de simulation déterministe, reproductible. Ce modèle de décision sera donc privilégié dans les parties suivantes.